### 函数测试

In [ ]:
import requests
from wisemodel_hub import file_download, lfs_file_download, snapshot_download, upload_with_progress_bar
from wisemodel_hub.parallel_download_with_resume import LFSDownload

repo_id_datasets = "renqibing/SafeMTData"
#repo_id_models = "CUHK-DVLab/Mini-Gemini-2B"
#repo_id_models = "IEIT-Yuan/model.safetensors"
repo_id_models = "zhanshijinwat/Steel-LLM"
file_name_models = "model.safetensors"
repo_id_codes = "OpenCoder/OpenCoder-llm"

#file_download(repo_id_datasets, "SafeMTData_1K.json")
#file_download(repo_id_datasets, "README.md")  # datasets_hub file
#file_download(repo_id_models, "config.json")  # model_hub file
# file_download(repo_id_codes, "README.md")       # code_hub file

#lfs_file_download(repo_id_datasets, "SafeMTData_1K.json")
#lfs_file_download(repo_id_models, "model-00001-of-00002.safetensors")
downloader = LFSDownload(repo_id_models, file_name_models, num_parts=8)
downloader.download()

### 测试获取文件大小

In [34]:
import requests
# not work
#url = "https://download.wisemodel.cn/file-proxy/CUHK-DVLab/Mini-Gemini-2B/-/raw/main/model-00001-of-00002.safetensors"
# ok
url = "https://download.wisemodel.cn/file-proxy/CUHK-DVLab/Mini-Gemini-2B/-/raw/main/model-00002-of-00002.safetensors"
# not work
#url = "https://download.wisemodel.cn/file-proxy/IEIT-Yuan/model.safetensors/-/raw/main/model.safetensors"
# ok
#url = "https://download.wisemodel.cn/file-proxy/zhanshijinwat/Steel-LLM/-/raw/main/model.safetensors"
res = requests.head(url)
res.raise_for_status()
print(int(res.headers.get("Content-Length", 0)))

2141794048


### 单独下载一个块测试

In [ ]:
from wisemodel_hub.parallel_download_with_resume import LFSDownload
from tqdm import tqdm

repo_id_models = "CUHK-DVLab/Mini-Gemini-2B"

downloader = LFSDownload(repo_id_models, "model-00002-of-00002.safetensors")
downloader.prepare()
downloader.progress_bar = tqdm(total=downloader.total_size, unit="B", unit_scale=True, desc=downloader.file_name)
downloader.download_part(535448512, 803172768, "/home/zhangjingbo/.cache/wisemodel/CUHK-DVLab_Mini-Gemini-2B/model-00002-of-00002.safetensors.part3")

### chatgpt写的lfs分块下载代码

In [7]:
import os
import requests
from concurrent.futures import ThreadPoolExecutor

def download_chunk(url, start_byte, end_byte, chunk_id, output_file, retries=3):
    """
    下载文件的一个分块，支持断点续传。
    """
    chunk_file = f"{output_file}.part{chunk_id}"
    
    # 如果文件块已经下载，跳过
    if os.path.exists(chunk_file) and os.path.getsize(chunk_file) == (end_byte - start_byte + 1):
        print(f"Chunk {chunk_id} already downloaded, skipping...")
        return

    headers = {'Range': f'bytes={start_byte}-{end_byte}'}
    try:
        # 下载当前块
        response = requests.get(url, headers=headers, stream=True)
        response.raise_for_status()

        # 保存文件块
        with open(chunk_file, 'wb') as f:
            f.write(response.content)

        print(f"Chunk {chunk_id} downloaded: {start_byte}-{end_byte}")

    except requests.RequestException as e:
        print(f"Error downloading chunk {chunk_id}: {e}")
        if retries > 0:
            print(f"Retrying chunk {chunk_id}...")
            download_chunk(url, start_byte, end_byte, chunk_id, output_file, retries-1)
        else:
            print(f"Failed to download chunk {chunk_id} after multiple retries.")
            raise e  # 如果多次失败，则抛出异常，导致整体失败

def get_file_size(url):
    """
    获取文件大小。
    """
    response = requests.head(url)
    response.raise_for_status()
    return int(response.headers['Content-Length'])

def merge_chunks(output_file, num_chunks):
    """
    合并下载的文件块。
    """
    with open(output_file, 'wb') as f:
        for i in range(num_chunks):
            chunk_file = f"{output_file}.part{i}"
            with open(chunk_file, 'rb') as cf:
                f.write(cf.read())
            os.remove(chunk_file)
    print(f"File {output_file} merged successfully!")

def download_file_in_chunks(url, output_file, num_chunks=8):
    """
    将文件分块并行下载，支持断点续传。
    """
    file_size = get_file_size(url)
    print(f"File size: {file_size} bytes")
    
    # 定义每块的大小
    chunk_size = file_size // num_chunks
    futures = []

    with ThreadPoolExecutor(max_workers=num_chunks) as executor:
        for i in range(num_chunks):
            start_byte = i * chunk_size
            # 最后一块可能会包含剩余字节
            end_byte = start_byte + chunk_size - 1 if i < num_chunks - 1 else file_size - 1

            futures.append(executor.submit(download_chunk, url, start_byte, end_byte, i, output_file))

        # 等待所有任务完成，任何失败都会触发整体失败
        for future in futures:
            future.result()

    # 合并所有分块
    merge_chunks(output_file, num_chunks)

if __name__ == "__main__":
    # 文件下载 URL
    url = "https://download.wisemodel.cn/file-proxy/CUHK-DVLab/Mini-Gemini-2B/-/raw/main/model-00002-of-00002.safetensors"
    # 输出文件名
    output_file = "model-00002-of-00002.safetensors"

    try:
        download_file_in_chunks(url, output_file)
        print("Download completed!")
    except Exception as e:
        print(f"Download failed: {e}")

File size: 2141794048 bytes
Error downloading chunk 4: ('Connection broken: IncompleteRead(0 bytes read, 267724256 more expected)', IncompleteRead(0 bytes read, 267724256 more expected))
Retrying chunk 4...
Error downloading chunk 7: ('Connection broken: IncompleteRead(0 bytes read, 267724256 more expected)', IncompleteRead(0 bytes read, 267724256 more expected))
Retrying chunk 7...
Error downloading chunk 3: ('Connection broken: IncompleteRead(0 bytes read, 267724256 more expected)', IncompleteRead(0 bytes read, 267724256 more expected))
Retrying chunk 3...
Error downloading chunk 5: ('Connection broken: IncompleteRead(0 bytes read, 267724256 more expected)', IncompleteRead(0 bytes read, 267724256 more expected))
Retrying chunk 5...
Error downloading chunk 6: ('Connection broken: IncompleteRead(0 bytes read, 267724256 more expected)', IncompleteRead(0 bytes read, 267724256 more expected))
Retrying chunk 6...
Error downloading chunk 4: ('Connection broken: IncompleteRead(0 bytes read, 

### snap_download函数测试


In [ ]:
from wisemodel_hub import snapshot_download